<a href="https://colab.research.google.com/github/Bassendiaye/mes_notebooks/blob/main/VGG19_Online.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split
from collections import Counter

In [ ]:
# ============================================================
# 2. DATASET PERSONNALISÉ
# ============================================================
class AlbumentationsImageFolder(Dataset):
    def __init__(self, file_paths, labels, transform=None):
        self.file_paths = file_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        image = np.array(Image.open(self.file_paths[idx]).convert("RGB"))
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image=image)["image"]

        return image, label

In [ ]:
# ------------------- Transformations dynamiques -------------------
AUGMENTATIONS = [
    A.NoOp(p=1),
    A.HorizontalFlip(p=1),
    A.VerticalFlip(p=1),
    A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=1),
    A.GridDistortion(num_steps=5, distort_limit=0.3, p=1),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=1),
    A.RGBShift(r_shift_limit=20, g_shift_limit=20, b_shift_limit=20, p=1),
    A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=20, val_shift_limit=15, p=1),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=1),
    A.Defocus(radius=(3, 5), p=1),
    A.MotionBlur(blur_limit=(3, 5), p=1),
    A.GaussianBlur(blur_limit=(3, 5), p=1)
]

train_transform = A.Compose([
    A.OneOf(AUGMENTATIONS, p=1),
    A.Resize(224, 224),
    A.Normalize(mean=(0.5,), std=(0.5,)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=(0.5,), std=(0.5,)),
    ToTensorV2()
])

In [ ]:

# ============================================================
# 4. LECTURE DU DATASET + SPLIT
# ============================================================
data_dir = "/content/drive/MyDrive/Dossier_de_Basse/Data_paper_TrainVal_Test/train_Val"

class_names = sorted(os.listdir(data_dir))
class_to_idx = {cls: i for i, cls in enumerate(class_names)}

file_paths = []
labels = []

for cls in class_names:
    cls_folder = os.path.join(data_dir, cls)
    for f in os.listdir(cls_folder):
        file_paths.append(os.path.join(cls_folder, f))
        labels.append(class_to_idx[cls])

# Train / Val
X_train, X_val, y_train, y_val = train_test_split(
    file_paths, labels,
    test_size=0.1,
    stratify=labels,
    random_state=42
)
print(Counter(y_train))
print(Counter(y_val))

In [ ]:
# ============================================================
# 5. WEIGHTED RANDOM SAMPLER
# ============================================================
class_counts = Counter(y_train)
total_count = sum(class_counts.values())

class_weights = {cls: total_count/class_counts[cls] for cls in class_counts}
sample_weights = [class_weights[label] for label in y_train]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

In [ ]:
# ============================================================
# 6. DATALOADERS
# ============================================================
train_dataset = AlbumentationsImageFolder(X_train, y_train, transform=train_transform)
val_dataset   = AlbumentationsImageFolder(X_val, y_val, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, sampler=sampler)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [ ]:
# ============================================================
# 7. VGG19
# ============================================================
num_classes = len(class_names)

vgg19 = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1)
vgg19.classifier[6] = nn.Linear(4096, num_classes)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vgg19 = vgg19.to(device)

In [ ]:
# ============================================================
# 8. LOSS + OPTIMIZER
# ============================================================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(vgg19.parameters(), lr=1e-4)

In [ ]:
# ============================================================
# 9. TRAIN & VALIDATION FUNCTIONS
# ============================================================
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0
    correct = 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()

    return running_loss / len(loader), correct / len(loader.dataset)


def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0
    correct = 0

    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)

            outputs = model(imgs)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()

    return running_loss / len(loader), correct / len(loader.dataset)

In [ ]:
# ============================================================
# 10. TRAINING LOOP + SAUVEGARDE MEILLEUR MODÈLE
# ============================================================
best_val_acc = 0.0
best_model_path = "/content/drive/MyDrive/Dossier_de_Basse/best_vgg19_online.pth"

EPOCHS = 25

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(vgg19, train_loader, criterion, optimizer)
    val_loss, val_acc = evaluate(vgg19, val_loader, criterion)

    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} - Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} - Val Acc: {val_acc:.4f}")

    # Sauvegarde du meilleur modèle
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(vgg19.state_dict(), best_model_path)
        print(f"✔ Nouveau meilleur modèle sauvegardé : {best_model_path} (Acc: {best_val_acc:.4f})")